# Évaluation cross-benchmark des modèles NLI 

In [ ]:
# Installation des bibliothèques nécessaires
!pip install -q transformers[torch] datasets evaluate accelerate optuna scikit-learn matplotlib pandas seaborn captum

# Vérification du GPU
import torch
print(f"GPU détecté : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN GPU'}")

In [ ]:
# Installation des librairies nécessaires
# À exécuter une seule fois dans Colab.

!pip -q install transformers datasets evaluate accelerate scikit-learn pandas matplotlib seaborn

In [ ]:
!pip install -q "datasets==3.6.0"

## 1. Imports et configuration générale

Cette partie importe les librairies et fixe les paramètres communs : longueur maximale des séquences, noms de classes et chemins de sortie.

In [ ]:
import os
import json
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

sns.set_theme(style="whitegrid")
plt.rcParams.update({"font.size": 11})

MAX_LENGTH = 128
OUTPUT_DIR = "./evaluation_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

LABEL_NAMES_3 = ["entailment", "neutral", "contradiction"]
LABEL_NAMES_HANS = ["entailment", "non-entailment"]

print("GPU disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nom GPU :", torch.cuda.get_device_name(0))

## 2. Chemins des modèles

Il faut indiquer les chemins vers les modèles entraînés.

Deux possibilités :
1. les modèles sont déjà dézippés dans Colab ou Drive ;
2. vous avez seulement les fichiers `.zip`, dans ce cas vous pouvez les dézipper avec la cellule suivante.

In [ ]:
# Chemins des modèles entraînés
# Ces chemins sont relatifs au dépôt GitHub.
# Modifiez-les si vous stockez vos modèles ailleurs.

MODELS = {
    "BERT_SNLI": "./models/final_bert_snli_model",
    "RoBERTa_SNLI": "./models/final_roberta_snli_model",
    "DeBERTa_SNLI": "./models/final_deberta_snli_model",
    "BERT_MNLI": "./models/final_bert_mnli_model",
    "BERT_ANLI": "./models/final_bert_anli_model",
}

for nom, chemin in MODELS.items():
    print(nom, "=>", chemin, "| existe :", os.path.exists(chemin))

## 3. Chargement des benchmarks

On charge :
- SNLI test : performance standard sur la même distribution que l’entraînement ;
- MNLI matched : généralisation sur des genres proches ;
- MNLI mismatched : généralisation sur des genres différents ;
- ANLI R1/R2/R3 : robustesse adversariale progressive ;
- HANS : test spécifique des heuristiques de chevauchement lexical, sous-séquence et constituants.

In [ ]:
def charger_benchmarks():
    benchmarks = {}

    print("Chargement SNLI test...")
    snli_test = load_dataset("snli", split="test")
    snli_test = snli_test.filter(lambda x: x["label"] != -1)
    benchmarks["SNLI_test"] = snli_test

    print("Chargement MNLI matched / mismatched...")
    benchmarks["MNLI_matched"] = load_dataset("glue", "mnli", split="validation_matched")
    benchmarks["MNLI_mismatched"] = load_dataset("glue", "mnli", split="validation_mismatched")

    print("Chargement ANLI R1 / R2 / R3...")
    benchmarks["ANLI_R1"] = load_dataset("anli", split="test_r1")
    benchmarks["ANLI_R2"] = load_dataset("anli", split="test_r2")
    benchmarks["ANLI_R3"] = load_dataset("anli", split="test_r3")

    print("Chargement HANS...")
    benchmarks["HANS"] = load_dataset("hans", split="validation", trust_remote_code=True)

    return benchmarks

benchmarks = charger_benchmarks()

for nom, ds in benchmarks.items():
    print(f"{nom:16s} : {len(ds)} exemples | colonnes : {ds.column_names}")

## 4. Fonctions utilitaires

Ces fonctions servent à :
- tokeniser les paires prémisse/hypothèse ;
- convertir les prédictions 3 classes en 2 classes pour HANS ;
- calculer les métriques ;
- afficher et sauvegarder les matrices de confusion ;
- analyser les performances par heuristique HANS.

In [ ]:
def tokeniser_dataset(dataset, tokenizer):
    def preprocess(examples):
        return tokenizer(
            examples["premise"],
            examples["hypothesis"],
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH
        )

    tokenized = dataset.map(preprocess, batched=True)
    return tokenized


def mapper_predictions_hans(predictions_3_classes):
    # Dans SNLI/MNLI/ANLI :
    # 0 = entailment, 1 = neutral, 2 = contradiction
    # Dans HANS :
    # 0 = entailment, 1 = non-entailment
    # Donc neutral + contradiction deviennent non-entailment.
    return np.array([0 if p == 0 else 1 for p in predictions_3_classes])


def calculer_metriques(y_true, y_pred, average="macro"):
    acc = accuracy_score(y_true, y_pred)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    return {
        "accuracy": acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
    }


def tracer_matrice_confusion(y_true, y_pred, labels_names, titre, save_path=None):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=labels_names,
        yticklabels=labels_names,
        cbar=False,
        annot_kws={"size": 12}
    )
    plt.title(titre, fontsize=13, pad=12, fontweight="bold")
    plt.ylabel("Classe réelle", fontweight="bold")
    plt.xlabel("Prédiction du modèle", fontweight="bold")
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")

    plt.show()


def rapport_par_heuristique_hans(dataset_hans, y_pred):
    df_hans = pd.DataFrame(dataset_hans)
    df_hans["pred"] = y_pred
    rows = []

    for heuristic in sorted(df_hans["heuristic"].unique()):
        sub = df_hans[df_hans["heuristic"] == heuristic]
        rows.append({
            "heuristic": heuristic,
            "n_examples": len(sub),
            "accuracy": accuracy_score(sub["label"], sub["pred"])
        })

    return pd.DataFrame(rows)

## 5. Fonction centrale d’évaluation

Cette fonction charge un modèle, évalue un dataset donné, affiche les scores, génère la matrice de confusion et retourne les résultats sous forme exploitable.

In [ ]:
def evaluer_modele_sur_dataset(nom_modele, chemin_modele, nom_dataset, dataset):
    print("\n" + "="*90)
    print(f"ÉVALUATION : {nom_modele} >>> {nom_dataset}")
    print("="*90)

    tokenizer = AutoTokenizer.from_pretrained(chemin_modele)
    model = AutoModelForSequenceClassification.from_pretrained(chemin_modele)

    tokenized_data = tokeniser_dataset(dataset, tokenizer)

    # On enlève les colonnes texte pour éviter les problèmes de format dans Trainer.
    colonnes_a_garder = {"input_ids", "attention_mask", "label"}
    colonnes_a_supprimer = [c for c in tokenized_data.column_names if c not in colonnes_a_garder]
    tokenized_data = tokenized_data.remove_columns(colonnes_a_supprimer)
    tokenized_data.set_format("torch")

    args = TrainingArguments(
        output_dir=os.path.join(OUTPUT_DIR, "tmp_trainer"),
        per_device_eval_batch_size=32,
        report_to=[],
        logging_strategy="no"
    )

    trainer = Trainer(model=model, args=args)

    predictions_output = trainer.predict(tokenized_data)
    logits = predictions_output.predictions
    predictions_3_classes = np.argmax(logits, axis=1)
    y_true = np.array(dataset["label"])

    if nom_dataset == "HANS":
        y_pred = mapper_predictions_hans(predictions_3_classes)
        labels_names = LABEL_NAMES_HANS
    else:
        y_pred = predictions_3_classes
        labels_names = LABEL_NAMES_3

    metriques = calculer_metriques(y_true, y_pred)

    print("\nMétriques principales :")
    for k, v in metriques.items():
        print(f"{k:20s}: {v:.4f}")

    print("\nRapport de classification :")
    rapport_txt = classification_report(
        y_true,
        y_pred,
        target_names=labels_names,
        digits=4,
        zero_division=0
    )
    print(rapport_txt)

    nom_fichier_base = (
        nom_modele.replace(" ", "_")
        .replace("é", "e")
        .replace("è", "e")
        .replace("î", "i")
        .replace("'", "")
        .replace("/", "_")
    )
    fig_path = os.path.join(OUTPUT_DIR, f"confusion_{nom_fichier_base}_{nom_dataset}.png")

    tracer_matrice_confusion(
        y_true,
        y_pred,
        labels_names,
        f"Matrice de confusion\n{nom_modele} sur {nom_dataset}",
        save_path=fig_path
    )

    hans_details = None
    if nom_dataset == "HANS":
        hans_details = rapport_par_heuristique_hans(dataset, y_pred)
        print("\nAnalyse HANS par heuristique :")
        display(hans_details)

    result_row = {
        "modele": nom_modele,
        "chemin_modele": chemin_modele,
        "dataset": nom_dataset,
        "n_examples": len(dataset),
        **metriques,
        "confusion_matrix_path": fig_path
    }

    rapport_dict = classification_report(
        y_true,
        y_pred,
        target_names=labels_names,
        output_dict=True,
        zero_division=0
    )

    details = {
        "result_row": result_row,
        "classification_report": rapport_dict,
        "hans_details": hans_details.to_dict(orient="records") if hans_details is not None else None
    }

    del model, trainer, tokenized_data
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return details

## 6. Boucle d’évaluation complète

Cette cellule teste chaque modèle sur chaque benchmark.  
Elle peut être longue, mais elle ne fait aucune phase d’entraînement.

In [ ]:
tous_les_resultats = []
tous_les_rapports = {}

for nom_modele, chemin_modele in MODELS.items():
    if not os.path.exists(chemin_modele):
        print(f"⚠️ Chemin introuvable pour {nom_modele} : {chemin_modele}")
        print("Ce modèle sera ignoré.")
        continue

    tous_les_rapports[nom_modele] = {}

    for nom_dataset, dataset in benchmarks.items():
        details = evaluer_modele_sur_dataset(
            nom_modele=nom_modele,
            chemin_modele=chemin_modele,
            nom_dataset=nom_dataset,
            dataset=dataset
        )

        tous_les_resultats.append(details["result_row"])
        tous_les_rapports[nom_modele][nom_dataset] = {
            "classification_report": details["classification_report"],
            "hans_details": details["hans_details"]
        }

print("\nToutes les évaluations sont terminées.")

## 7. Tableau final des résultats

On transforme tous les résultats en tableau pour comparer BERT et RoBERTa sur les différents benchmarks.

In [ ]:
df_resultats = pd.DataFrame(tous_les_resultats)

colonnes_principales = [
    "modele", "dataset", "n_examples",
    "accuracy", "f1_macro", "precision_macro", "recall_macro",
    "f1_weighted", "precision_weighted", "recall_weighted"
]

df_resultats = df_resultats[colonnes_principales + ["chemin_modele", "confusion_matrix_path"]]
display(df_resultats[colonnes_principales])

csv_path = os.path.join(OUTPUT_DIR, "resultats_evaluation_mnli_anli_hans.csv")
json_path = os.path.join(OUTPUT_DIR, "resultats_evaluation_mnli_anli_hans.json")
reports_path = os.path.join(OUTPUT_DIR, "rapports_classification_details.json")

df_resultats.to_csv(csv_path, index=False)

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(tous_les_resultats, f, indent=4, ensure_ascii=False)

with open(reports_path, "w", encoding="utf-8") as f:
    json.dump(tous_les_rapports, f, indent=4, ensure_ascii=False)

print("CSV sauvegardé :", csv_path)
print("JSON sauvegardé :", json_path)
print("Rapports détaillés sauvegardés :", reports_path)

## 8. Graphiques comparatifs

Ces graphiques permettent de visualiser rapidement :
- l’accuracy par modèle et benchmark ;
- le macro-F1 par modèle et benchmark ;
- la chute de performance entre SNLI, MNLI et ANLI.

In [ ]:
if len(df_resultats) > 0:
    plt.figure(figsize=(12, 5))
    sns.barplot(data=df_resultats, x="dataset", y="accuracy", hue="modele")
    plt.title("Comparaison des modèles - Accuracy")
    plt.xticks(rotation=30, ha="right")
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "comparaison_accuracy.png"), dpi=150, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(12, 5))
    sns.barplot(data=df_resultats, x="dataset", y="f1_macro", hue="modele")
    plt.title("Comparaison des modèles - Macro-F1")
    plt.xticks(rotation=30, ha="right")
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "comparaison_macro_f1.png"), dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Aucun résultat disponible à afficher.")

## 9. Analyse de la baisse de performance

On calcule la différence entre SNLI test et les autres benchmarks.  
Une forte baisse sur MNLI indique une difficulté de généralisation.  
Une forte baisse sur ANLI ou HANS indique une fragilité face aux exemples adversariaux ou aux heuristiques.

In [ ]:
if len(df_resultats) > 0 and "SNLI_test" in df_resultats["dataset"].unique():
    rows = []

    for modele in df_resultats["modele"].unique():
        sub = df_resultats[df_resultats["modele"] == modele]
        snli_acc = sub.loc[sub["dataset"] == "SNLI_test", "accuracy"]
        snli_f1 = sub.loc[sub["dataset"] == "SNLI_test", "f1_macro"]

        if len(snli_acc) == 0:
            continue

        snli_acc = float(snli_acc.iloc[0])
        snli_f1 = float(snli_f1.iloc[0])

        for _, row in sub.iterrows():
            rows.append({
                "modele": modele,
                "dataset": row["dataset"],
                "accuracy": row["accuracy"],
                "delta_accuracy_vs_SNLI": row["accuracy"] - snli_acc,
                "f1_macro": row["f1_macro"],
                "delta_f1_vs_SNLI": row["f1_macro"] - snli_f1,
            })

    df_delta = pd.DataFrame(rows)
    display(df_delta)

    delta_path = os.path.join(OUTPUT_DIR, "delta_performance_vs_snli.csv")
    df_delta.to_csv(delta_path, index=False)
    print("Tableau des écarts sauvegardé :", delta_path)

    plt.figure(figsize=(12, 5))
    sns.barplot(data=df_delta[df_delta["dataset"] != "SNLI_test"], x="dataset", y="delta_f1_vs_SNLI", hue="modele")
    plt.axhline(0, color="black", linewidth=1)
    plt.title("Chute du Macro-F1 par rapport à SNLI test")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "delta_macro_f1_vs_snli.png"), dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Impossible de calculer les écarts : SNLI_test absent ou aucun résultat.")

## 10. Télécharger les résultats depuis Colab

Si Drive ne fonctionne pas, cette cellule compresse le dossier de sortie et télécharge les fichiers sur ton PC.

In [ ]:
# À exécuter à la fin si vous êtes sur Colab et que Drive ne fonctionne pas.

# !zip -r evaluation_outputs.zip ./evaluation_outputs

# files.download("evaluation_outputs.zip")

In [ ]:
# Optionnel : copier les sorties vers un espace de sauvegarde externe si nécessaire.

# Interprétation attendue

Dans le rapport, vous pourrez utiliser cette logique :

- SNLI test mesure la performance standard après entraînement sur SNLI.
- MNLI matched/mismatched mesure la généralisation hors domaine.
- ANLI R1/R2/R3 mesure la robustesse adversariale.
- HANS mesure la sensibilité aux heuristiques superficielles, notamment le chevauchement lexical.

Une chute forte entre SNLI et ANLI/HANS indique que le modèle a appris certaines régularités de SNLI, mais reste fragile face aux exemples construits pour piéger les biais.